In [1]:
%cd /kaggle/working

!git clone -b experiment2_polyphonicdrums_newwork https://github.com/Wusat12/ADT.git

/kaggle/working
Cloning into 'ADT'...
remote: Enumerating objects: 358, done.
remote: Counting objects: 100% (358/358), done.
remote: Compressing objects: 100% (243/243), done.
remote: Total 358 (delta 132), reused 336 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (358/358), 10.56 MiB | 27.09 MiB/s, done.
Resolving deltas: 100% (132/132), done.


In [1]:
%pip install -q partitura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.9/363.9 kB 5.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.9/261.9 kB 14.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import ast
import shutil
import sys
import torch

repo = Path("/kaggle/working/ADT")
eval_path = repo / "evaluate.py"
run_path = repo / "run.py"

eval_source = eval_path.read_text(encoding="utf-8")
run_source = run_path.read_text(encoding="utf-8")

# --------------------------------------------------
# 1. Extend evaluate_model without changing its behavior
#    for existing callers.
# --------------------------------------------------

old_signature = (
    "def evaluate_model(model: torch.nn.Module, "
    "test_loader: DataLoader, transforms: Compose, "
    "device: str, seed: int = None):"
)

new_signature = (
    "def evaluate_model(model: torch.nn.Module, "
    "test_loader: DataLoader, transforms: Compose, "
    "device: str, seed: int = None, return_details: bool = False):"
)

if new_signature not in eval_source:
    assert eval_source.count(old_signature) == 1, (
        "Unexpected evaluate_model signature. No changes made."
    )
    eval_source = eval_source.replace(
        old_signature, new_signature, 1
    )

old_return = "    return f_measure(predictions)"

new_return = """    if return_details:
        return detailed_metrics(predictions)

    return f_measure(predictions)"""

if new_return not in eval_source:
    assert eval_source.count(old_return) == 1, (
        "Unexpected evaluation return. No changes made."
    )
    eval_source = eval_source.replace(old_return, new_return, 1)

# --------------------------------------------------
# 2. Add precision, recall, F1 and TP/FP/FN reporting.
# --------------------------------------------------

details_function = '''
def detailed_metrics(predictions: torch.Tensor) -> dict:
    """Metrics from existing onset-matching TP, FP and FN counts."""

    counts = predictions.detach().cpu().to(torch.float64)

    if counts.shape != (3, 3):
        raise ValueError(
            f"Expected [3, 3] TP/FP/FN counts, got {tuple(counts.shape)}"
        )

    names = ("bass_drum", "snare_drum", "hi_hat")

    def calculate(tp, fp, fn):
        precision = tp / (tp + fp) if tp + fp else 1.0
        recall = tp / (tp + fn) if tp + fn else 1.0
        f1 = (
            2.0 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        return {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
        }

    classes = {}

    for index, name in enumerate(names):
        tp, fp, fn = [
            float(value) for value in counts[index]
        ]

        classes[name] = {
            **calculate(tp, fp, fn),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
        }

    tp, fp, fn = [
        float(value) for value in counts.sum(dim=0)
    ]

    micro = {
        **calculate(tp, fp, fn),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
    }

    macro = {
        metric: sum(
            classes[name][metric] for name in names
        ) / len(names)
        for metric in ("precision", "recall", "f1")
    }

    # Check agreement with the existing F1 implementation.
    original_micro, original_macro, original_classes = (
        f_measure(predictions)
    )

    assert abs(micro["f1"] - original_micro.item()) < 1e-5
    assert abs(macro["f1"] - original_macro.item()) < 1e-5

    for index, name in enumerate(names):
        assert abs(
            classes[name]["f1"] - original_classes[index].item()
        ) < 1e-5

    return {
        "micro": micro,
        "macro": macro,
        "classes": classes,
        "onset_tolerance_frames": 5,
    }


'''

if "def detailed_metrics(" not in eval_source:
    marker = "def f_measure(predictions: torch.Tensor):"

    assert eval_source.count(marker) == 1, (
        "Cannot locate f_measure. No changes made."
    )

    eval_source = eval_source.replace(
        marker, details_function + marker, 1
    )

# --------------------------------------------------
# 3. Request detailed metrics for test evaluation.
# --------------------------------------------------

old_call = '''        test_metrics = evaluate_model(
            evaluation_model,
            test_loader,
            test_model_transform,
            device=device,
        )'''

new_call = '''        test_metrics = evaluate_model(
            evaluation_model,
            test_loader,
            test_model_transform,
            device=device,
            return_details=True,
        )'''

if new_call not in run_source:
    assert run_source.count(old_call) == 1, (
        "Unexpected test evaluation call. No changes made."
    )

    run_source = run_source.replace(
        old_call, new_call, 1
    )

# --------------------------------------------------
# 4. Check syntax and verify numerical consistency.
# --------------------------------------------------

ast.parse(eval_source)
ast.parse(run_source)

namespace = {"torch": torch}

# Test the modified evaluator code in memory first.
exec(compile(eval_source, str(eval_path), "exec"), namespace)

sample_counts = torch.tensor([
    [90., 10., 20.],
    [70., 15., 25.],
    [80., 20., 10.],
])

result = namespace["detailed_metrics"](sample_counts)

assert result["micro"]["tp"] == 240
assert result["micro"]["fp"] == 45
assert result["micro"]["fn"] == 55
assert len(result["classes"]) == 3

# --------------------------------------------------
# 5. Back up and save only after checks pass.
# --------------------------------------------------

for path in (eval_path, run_path):
    backup = path.with_name(path.name + ".before_detailed_metrics")

    if not backup.exists():
        shutil.copy2(path, backup)

eval_path.write_text(eval_source, encoding="utf-8")
run_path.write_text(run_source, encoding="utf-8")

print("✅ Evaluation patch installed.")
print("✅ Micro precision, recall and F1.")
print("✅ Macro precision, recall and F1.")
print("✅ BD, SD and HH precision, recall and F1.")
print("✅ TP, FP and FN counts.")
print("✅ Original F1 calculation numerically verified.")
print("✅ JSON-compatible results.")
print("✅ Both Python files passed syntax checks.")
print("✅ No training started.")

✅ Evaluation patch installed.
✅ Micro precision, recall and F1.
✅ Macro precision, recall and F1.
✅ BD, SD and HH precision, recall and F1.
✅ TP, FP and FN counts.
✅ Original F1 calculation numerically verified.
✅ JSON-compatible results.
✅ Both Python files passed syntax checks.
✅ No training started.


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import zipfile
import base64
import os

# Your existing complete backup
zip_path = Path("/kaggle/working/ADT_FINAL_BACKUP.zip")

assert zip_path.is_file(), f"Backup not found: {zip_path}"

# Verify ZIP integrity
with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None, "ZIP is corrupted!"

    names = archive.namelist()

    # Verify corrected Log-Mel models and results
    for fold in [1, 2, 3]:
        prefix = (
            "ADT_CV/ADT/experiments/vogl_cbgru_b/"
            "logmel/Architecture/idmt_logmel/IDMT/"
            f"fold_{fold}/"
        )

        assert prefix + "model.pt" in names
        assert prefix + "test_results.json" in names

print("✅ Complete ZIP verified")
print("✅ All 3 Log-Mel models included")
print("✅ All 3 Log-Mel results included")
print("✅ Updated code and previous archived experiments preserved")

size_mb = zip_path.stat().st_size / (1024 * 1024)

print(f"\nBackup: {zip_path.name}")
print(f"Size: {size_mb:.2f} MB")

# Same Base64 download approach as your original script.
# WARNING: 94 MB becomes approximately 126 MB of Base64 data.
print("\nGenerating direct download link...")

encoded = base64.b64encode(
    zip_path.read_bytes()
).decode("ascii")

display(HTML(
    f'<a href="data:application/zip;base64,{encoded}" '
    f'download="{zip_path.name}" '
    f'style="font-size:18px;font-weight:bold;'
    f'color:#2563eb;">'
    f'⬇ Download Complete ADT Backup ZIP ({size_mb:.2f} MB)'
    f'</a>'
))

del encoded

✅ Complete ZIP verified
✅ All 3 Log-Mel models included
✅ All 3 Log-Mel results included
✅ Updated code and previous archived experiments preserved

Backup: ADT_FINAL_BACKUP.zip
Size: 94.26 MB

Generating direct download link...
